# Test main script function: Add Reference

In [1]:
# Script to take folder of images and database (sql and vector)
from ultralytics import YOLO
from torch import mean, hub, nn, cuda, no_grad, norm, tensor, stack
from pymilvus import connections, Collection, utility
from uuid import uuid4

from torch.utils.data import DataLoader, Dataset, TensorDataset
from concurrent.futures import ThreadPoolExecutor

import ring
import detect
import ntfy
import database_vec
import database_sql

from gc import collect

ImportError: cannot import name 'create_new_collection' from partially initialized module 'database_vec' (most likely due to a circular import) (/mnt/nvm/repos/ring_detector/database_vec.py)

In [2]:
model_path = './models/yolov8m.pt'

# Load a REGULAR model
reg_model = YOLO(model_path, 'detect')  # pretrained YOLOv8n model
reg_model.to('cuda:0')
reg_model.name = 'yolov8'

# Load EMBEDDING model
emb_model = YOLO(model_path, 'detect')  # pretrained YOLOv8n model
emb_model.to('cuda:0')
emb_model.model.model = emb_model.model.model[:-1]
emb_model.name = "emb-yolov8"

# Ref images path
ref_input_path = './training_vehicle'

# resnet models
resnet_model = hub.load('pytorch/vision:v0.10.0', 'resnet50', pretrained=True)
resnet_model = nn.Sequential(*(list(resnet_model.children())[:-1]))
resnet_model.eval()
resnet_model.to('cuda:0')
resnet_model.name = 'resnet50'

Using cache found in /home/superdave/.cache/torch/hub/pytorch_vision_v0.10.0
/mnt/nvm/repos/ring_detector/.ringenv/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/mnt/nvm/repos/ring_detector/.ringenv/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
mv_conn = connections.connect(
alias="default",
host='172.21.0.5',
port='19530'
)

In [ ]:
utility.drop_collection("reference")

In [ ]:
# create reference Collection
new_collection = "reference"
database_vec.create_milvus_collection(new_collection)


In [ ]:
reference_collection = Collection('reference')

In [ ]:
# setup indexing for the collection
database_vec.create_milvus_index(reference_collection)

### Test Model Embedding speeds

In [3]:
# Get Image embeddings of the reference images - YOLOv8
image_paths, ref_img_embeddings, ref_vector = detect.get_embeddings(ref_input_path, emb_model, if_refence=True)

 ^^^^^^^^ ----------- Function 'get_embeddings' executed in 2.3241s --------- ^^^^^^^^^^^^^


In [4]:
len(ref_vector)

576

In [5]:
ref_vector

[0.026859046891331673,
 0.021249866113066673,
 0.0037175556644797325,
 -0.026432767510414124,
 0.06306134164333344,
 -0.01879010908305645,
 0.013398168608546257,
 0.05031919106841087,
 0.03170360252261162,
 -0.04144864156842232,
 0.006283950991928577,
 -0.017785727977752686,
 0.040287137031555176,
 0.047069355845451355,
 0.0331176221370697,
 -0.017067013308405876,
 0.046757813543081284,
 -0.052207961678504944,
 -0.03392577916383743,
 -0.032655004411935806,
 0.022627925500273705,
 -0.003949647303670645,
 -0.013088181614875793,
 -0.030059387907385826,
 -0.0005968641489744186,
 -0.0024588739033788443,
 0.043740179389715195,
 -0.006358692888170481,
 0.007605380844324827,
 -0.02348916605114937,
 -0.035635676234960556,
 -0.014236696064472198,
 -0.06251750141382217,
 -0.02424663119018078,
 0.08646659553050995,
 0.05970676243305206,
 -0.03303743898868561,
 0.0007757276180200279,
 0.009273829869925976,
 0.016681265085935593,
 0.053435068577528,
 -0.05822105333209038,
 -0.014273208566009998,
 0.

In [5]:
# clean up after first model
del image_paths, ref_img_embeddings, ref_vector
collect()
cuda.empty_cache()

testing the resnet functionality

In [6]:
# Get Image embeddings of the reference images - ResNet
image_paths, ref_img_embeddings, ref_vector = detect.get_embeddings(ref_input_path, resnet_model, if_refence=True)

In [7]:
ref_img_embeddings[0]

[0.023165719583630562,
 0.006484502926468849,
 0.007252541836351156,
 0.014670087024569511,
 0.009722364135086536,
 0.005295867566019297,
 0.021235408261418343,
 0.0035542361438274384,
 0.019597021862864494,
 0.007588776759803295,
 0.006900291424244642,
 0.0077657559886574745,
 0.007599189877510071,
 0.01635640859603882,
 0.013099978677928448,
 0.02021966315805912,
 0.02690458670258522,
 0.011934704147279263,
 0.006838710978627205,
 0.014162346720695496,
 0.02436213567852974,
 0.024442240595817566,
 0.005712153855711222,
 0.03139009699225426,
 0.01605873554944992,
 0.023659195750951767,
 0.009946531616151333,
 0.016022520139813423,
 0.00204124441370368,
 0.011371782049536705,
 0.002351402072235942,
 0.04235348105430603,
 0.015409440733492374,
 0.004757218528538942,
 0.02169996313750744,
 0.00335456570610404,
 0.02180376835167408,
 0.035167645663022995,
 0.03772427514195442,
 0.0057695526629686356,
 0.026590442284941673,
 0.017789894714951515,
 0.009447303600609303,
 0.00530692050233483

In [9]:
len(ref_vector)

2048

### Setup for databasing

In [ ]:
# create list of UUIDs for database
ref_uuid_list = [uuid4() for _ in range(len(image_paths))]

In [ ]:
# database the vectors to Milvus
reference_collection.insert([ref_uuid_list, image_paths, ref_img_embeddings])

In [ ]:
# send notification of new reference entry
ntfy.new_ref_ntfy(f'Add new vehicle!')

In [ ]:
# database reference to postgres
database_sql.insert_reference('reference','cleaners_car', str(ref_vector))

In [ ]:
# query the collection for the closest reference image to the ref_vector
reference_collection.load()

search_results = database_vec.milvus_similarity_search(reference_collection, ref_vector, 1)

In [ ]:
search_results[0].ids[0]

In [ ]:
search_results[0].distances

In [ ]:
hit = search_results[0][0]

In [ ]:
search_results[0][0].entity.get('img_path')

In [ ]:
hit.entity.get('embedding')

In [ ]:
hit.entity.get('img_path')

In [ ]:
hit.entity.get('uuid')